# Lab 09-12 · Advanced Transistors

Companion to chapter [09-12](../../chapters/09-12-finfet-gaa-cfet-backside-power.md). Handout: [README.md](README.md).

Potentials are in volts relative to the long-channel center potential; lengths in nm; $\varepsilon_s/\varepsilon_{ox}$ = 3.

In [1]:
import math
import numpy as np
from scipy.optimize import brentq
from scipy.sparse import diags, lil_matrix
from scipy.sparse.linalg import spsolve
from scipy.linalg import solve_banded
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def lam_dg(t, tox):
    return math.sqrt(3 * t * tox / 2 + t * t / 8)

def center_potential(L, lam, vds, a=0.2, n=2001):
    # finite-difference solution of u'' = u/lam^2 with u(0) = a, u(L) = a + vds
    x = np.linspace(0, L, n); h = x[1] - x[0]; m = n - 2
    A = diags([np.full(m - 1, 1 / h**2), np.full(m, -2 / h**2 - 1 / lam**2), np.full(m - 1, 1 / h**2)], [-1, 0, 1], format="csc")
    rhs = np.zeros(m); rhs[0] -= a / h**2; rhs[-1] -= (a + vds) / h**2
    return x, np.concatenate(([a], spsolve(A, rhs), [a + vds]))

def dibl(L, lam):
    return (center_potential(L, lam, 0.7)[1].min() - center_potential(L, lam, 0.05)[1].min()) / 0.65

## Part 1 — The barrier and DIBL

*Predict first:* at $L$ = 20 nm, by what factor does DIBL fall when the body thins from 7 nm to 5 nm?

In [2]:
l7, l5 = lam_dg(7, 1), lam_dg(5, 1)
d7, d5 = dibl(20, l7), dibl(20, l5)
est = math.exp(20 / (2 * l5) - 20 / (2 * l7))
print(f"lambda: 7 nm {l7:.3f} nm, 5 nm {l5:.3f} nm; DIBL {d7*1e3:.1f} and {d5*1e3:.1f} mV/V; ratio {d7/d5:.2f} (estimate {est:.2f})")
assert abs(d7 * 1e3 - 51.6) < 0.1 and abs(d5 * 1e3 - 28.5) < 0.1 and abs(d7 / d5 - 1.81) < 5e-3 and abs(est - 1.85) < 5e-3
L50 = brentq(lambda L: dibl(L, l7) - 0.050, 12, 60, xtol=1e-4)
print(f"7 nm body reaches 50 mV/V at L = {L50:.1f} nm")
assert abs(L50 - 20.3) < 0.05
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for vds in (0.05, 0.7):
    x, u = center_potential(20, l7, vds)
    ax[0].plot(x, u, label=f"V_DS = {vds} V")
ax[0].set_xlabel("x (nm)"); ax[0].set_ylabel("u = phi_c - V_G' (V)"); ax[0].legend()
Ls = np.linspace(12, 40, 40)
for lam, lab in ((l7, "DG 7 nm"), (l5, "DG 5 nm")):
    ax[1].semilogy(Ls, [dibl(L, lam) * 1e3 for L in Ls], label=lab)
ax[1].set_xlabel("L (nm)"); ax[1].set_ylabel("DIBL (mV/V)"); ax[1].legend(); plt.show()

lambda: 7 nm 4.077 nm, 5 nm 3.260 nm; DIBL 51.6 and 28.5 mV/V; ratio 1.81 (estimate 1.85)
7 nm body reaches 50 mV/V at L = 20.3 nm


## Part 2 — Laplace's equation in two dimensions

Grid 0.1 nm; silicon ($\varepsilon$ = 3) between two 1 nm oxides ($\varepsilon$ = 1); gates at 0 V; source and drain edges at $a$ and $a + V_{DS}$ in the silicon, falling linearly to the gate potential across the oxide.

*Predict first:* more or less barrier lowering than the parabolic model?

In [3]:
def solve2d(L, t, tox, vds, a=0.2, h=0.1):
    H = t / 2 + tox
    nx, ny = int(round(L / h)) + 1, int(round(2 * H / h)) + 1
    y = np.linspace(-H, H, ny)
    eps = np.where(np.abs(y) <= t / 2 + 1e-9, 3.0, 1.0)
    N = nx * ny
    A = lil_matrix((N, N)); b = np.zeros(N)
    k = lambda i, j: i * ny + j
    for i in range(nx):
        for j in range(ny):
            r = k(i, j)
            if j in (0, ny - 1):
                A[r, r] = 1.0; continue
            if i in (0, nx - 1):
                vb = a if i == 0 else a + vds
                yy = abs(y[j])
                A[r, r] = 1.0; b[r] = vb if yy <= t / 2 else vb * (H - yy) / tox; continue
            eN = 2 * eps[j] * eps[j + 1] / (eps[j] + eps[j + 1])
            eS = 2 * eps[j] * eps[j - 1] / (eps[j] + eps[j - 1])
            A[r, r] = -(eN + eS + 2 * eps[j])
            A[r, k(i, j + 1)] = eN; A[r, k(i, j - 1)] = eS
            A[r, k(i + 1, j)] = eps[j]; A[r, k(i - 1, j)] = eps[j]
    phi = spsolve(A.tocsr(), b).reshape(nx, ny)
    return phi[:, ny // 2]

for t in (7, 5):
    u_lo, u_hi = solve2d(20, t, 1, 0.05).min(), solve2d(20, t, 1, 0.7).min()
    lam = lam_dg(t, 1)
    m_lo, m_hi = center_potential(20, lam, 0.05)[1].min(), center_potential(20, lam, 0.7)[1].min()
    print(f"t = {t} nm: 2D barrier lowering {u_lo*1e3:.1f}/{u_hi*1e3:.1f} mV, model {m_lo*1e3:.1f}/{m_hi*1e3:.1f} mV; "
          f"DIBL 2D {(u_hi-u_lo)/0.65*1e3:.1f}, model {(m_hi-m_lo)/0.65*1e3:.1f} mV/V")
    assert abs(u_hi / m_hi - 1) < 0.05 and abs(u_lo / m_lo - 1) < 0.05

t = 7 nm: 2D barrier lowering 37.7/71.0 mV, model 38.2/71.8 mV; DIBL 2D 51.2, model 51.6 mV/V


t = 5 nm: 2D barrier lowering 20.3/38.4 mV, model 20.8/39.3 mV; DIBL 2D 27.9, model 28.5 mV/V


## Part 3 — Width per footprint

*Predict first:* how many 50 nm tall, 6 nm wide fins match three 30 nm × 6 nm sheets?

In [4]:
fin = lambda h, w: 2 * h + w
stack = lambda n, w, t: n * 2 * (w + t)
print(f"fin {fin(50, 6)} nm; stack {stack(3, 30, 6)} nm = {stack(3, 30, 6)/fin(50, 6):.2f} fins")
assert abs(stack(3, 30, 6) / fin(50, 6) - 2.04) < 5e-3
for n in (1, 2, 3, 4, 5):
    print(f"{n} sheets of 30 nm x 6 nm: W_eff = {stack(n, 30, 6)} nm")

fin 106 nm; stack 216 nm = 2.04 fins
1 sheets of 30 nm x 6 nm: W_eff = 72 nm
2 sheets of 30 nm x 6 nm: W_eff = 144 nm
3 sheets of 30 nm x 6 nm: W_eff = 216 nm
4 sheets of 30 nm x 6 nm: W_eff = 288 nm
5 sheets of 30 nm x 6 nm: W_eff = 360 nm


## Part 4 — Power rails

A rail of resistance $R$ split into many segments, with the load current drawn evenly along it.

*Predict first:* both ends against one end?

In [5]:
def ladder_drop(R, I, n=20000, both=True):
    g = n / R
    m = n - 1 if both else n
    ab = np.zeros((3, m)); ab[0, 1:] = -g; ab[1] = 2 * g; ab[2, :-1] = -g
    if not both:
        ab[1, -1] = g
    load = np.full(m, I / (n - 1))
    if not both:
        load[-1] = 0.0
    return solve_banded((1, 1), ab, load).max()
R_front = 5e-6 * 10e-4 / (20e-7 * 40e-7)
R_back = 3e-6 * 10e-4 / (200e-7 * 200e-7)
both, one = ladder_drop(R_front, 0.2e-3), ladder_drop(R_front, 0.2e-3, both=False)
print(f"front-side rail {R_front:.0f} ohm: {both*1e3:.2f} mV (both ends), {one*1e3:.2f} mV (one end); "
      f"backside rail {R_back:.2f} ohm: {ladder_drop(R_back, 0.2e-3)*1e3:.3f} mV")
assert abs(both * 1e3 - 15.6) < 0.05 and abs(one * 1e3 - 62.5) < 0.05

front-side rail 625 ohm: 15.63 mV (both ends), 62.50 mV (one end); backside rail 7.50 ohm: 0.188 mV
